In [ ]:
siamese_accuracies = []
siamese_precisions = []
siamese_recalls = []
siamese_f1s = []

diplomski_accuracies = []
diplomski_precisions = []
diplomski_recalls = []
diplomski_f1s = []

ExoplANNET_accuracies = []
ExoplANNET_precisions = []
ExoplANNET_recalls = []
ExoplANNET_f1s = []

In [ ]:
for seed in range(42, 47):
    def seed_everything(seed):
        import random
        import torch
        import numpy as np
    
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True
        
    seed_everything(seed)



    import pandas as pd
    import numpy as np
    import re
    from astropy.io import fits
    import matplotlib.pyplot as plt

    
    
    def parse_image(image_str):
        image_str = image_str.strip('[] ')
        
        rows = re.split(r'\]\s*\[', image_str)
        array = []
        for row in rows:
            elements = row.split()
            array.append([float(e) for e in elements])
        return np.array(array)
    
    # harps_images = pd.DataFrame(pd.read_csv('/kaggle/input/datasets/maanav0114/harps-images/data.csv', converters={'image': parse_image}))
    # harps_images = pd.DataFrame(pd.read_pickle('/kaggle/input/datasets/maanav0114/harps-images/HARPS_data_with_candidates_and_phase.pkl'))
    harps_images = pd.DataFrame(pd.read_pickle('data/HARPS_data_with_candidates_and_phase_random_shift.pkl'))
    # harps_images = pd.DataFrame(pd.read_csv('/kaggle/input/datasets/maanav0114/harps-images/HARPS_GAFs_with_Candidates.csv', converters = {'image': parse_image}))
    # hires_images = pd.DataFrame(pd.read_csv('/kaggle/input/datasets/maanav0114/harps-images/HIRES_GAFs.csv', converters={'image': parse_image}))
    # hires_images = pd.DataFrame(pd.read_pickle('/kaggle/input/datasets/maanav0114/harps-images/HIRES_data_with_candidates_and_phase.pkl'))
    hires_images = pd.DataFrame(pd.read_pickle('data/HIRES_data_with_candidates_and_phase_random_shift.pkl'))
    # hires_images = pd.DataFrame(pd.read_csv('/kaggle/input/datasets/maanav0114/harps-images/HIRES_GAFs_with_candidates.csv', converters = {'image': parse_image}))
    # hires_df = pd.read_csv('/kaggle/input/datasets/maanav0114/harps-n-dataset/preprocessed_HIRES_data.csv')
    hires_df = pd.read_csv('data/preprocessed_HIRES_data.csv')
    # with fits.open('/kaggle/input/datasets/maanav0114/harps-n-dataset/ADP.2023-12-04T15_16_53.464.fits') as data:
    with fits.open('data/ADP.2023-12-04T15_16_53.464.fits') as data:
        # Convert to native byte order immediately
            fits_data = data[1].data
            
            # Create DataFrame with native byte order arrays
            df_dict = {}
            for name in fits_data.dtype.names:
                arr = fits_data[name]
                # Convert to native byte order
                if not arr.dtype.isnative:
                    arr = arr.astype(arr.dtype.newbyteorder('='))
                df_dict[name] = arr
            
            harps_df = pd.DataFrame(df_dict)
    
    
    
    
    
    from torchvision import transforms
    
    
    
    transform = transforms.Compose([
        transforms.ToTensor(),
    ])
    
    
    
    
    
    import torch
    from torch.utils.data import DataLoader, Dataset
    from sklearn.model_selection import train_test_split
    import random
    
    
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    
    positives = harps_images[harps_images['label'] == 1].sample(n=(len(harps_images[harps_images['label'] == 1])))
    negatives = harps_images[harps_images['label'] == 0].sample(n=(len(harps_images[harps_images['label'] == 1])))
    
    train_pos, temp_pos = train_test_split(positives, test_size=0.3, random_state=seed)
    val_pos, test_pos = train_test_split(temp_pos, test_size=0.5, random_state=seed)  
    
    train_neg, temp_neg = train_test_split(negatives, test_size=0.3, random_state=seed)
    val_neg, test_neg = train_test_split(temp_neg, test_size=0.5, random_state=seed)  
    
    
    
    hires_positives = hires_images[hires_images['label'] == 1].sample(n=(len(hires_images[hires_images['label'] == 1])))
    hires_negatives = hires_images[hires_images['label'] == 0].sample(n=(len(hires_images[hires_images['label'] == 1])))
    
    hires_train_pos, hires_temp_pos = train_test_split(hires_positives, test_size=0.3, random_state=seed)
    hires_val_pos, hires_test_pos = train_test_split(hires_temp_pos, test_size=0.5, random_state=seed)  
    
    hires_train_neg, hires_temp_neg = train_test_split(hires_negatives, test_size=0.3, random_state=seed)
    hires_val_neg, hires_test_neg = train_test_split(hires_temp_neg, test_size=0.5, random_state=seed)
    
    
    
    train_data = np.array((pd.concat([train_pos, train_neg, hires_train_pos, hires_train_neg]))['image'])
    val_data = np.array((pd.concat([val_pos, val_neg, hires_val_pos, hires_val_neg]))['image'])
    test_data = pd.concat([test_pos, test_neg, hires_test_pos, hires_test_neg])
    
    
    
    class TripletDataset(Dataset):
        def __init__(self, triplets):
            self.triplets = triplets
    
        def __len__(self):
            return len(self.triplets)
    
        def __getitem__(self, idx):
            anchor, pos, neg = self.triplets[idx]
            
            return anchor, pos, neg
    
    
    
    
    class ImageDataset(Dataset):
        def __init__(self, images, labels):
            self.data = images
            self.labels = labels
    
        def __len__(self):
            return len(self.data)
    
        def __getitem__(self, idx):
            image = torch.from_numpy(self.data[idx])
            label = torch.tensor(self.labels[idx])
    
            return image, label
    
    
    train_labels = []
    val_labels = []
    test_labels = []
    
    for i in range(len(train_pos) + len(hires_train_pos)):
        train_labels.append(1)
    for i in range(len(train_neg) + len(hires_train_neg)):
        train_labels.append(0)
    
    for i in range(len(val_pos) + len(hires_val_pos)):
        val_labels.append(1)
    for i in range(len(val_neg) + len(hires_val_neg)):
        val_labels.append(0)
    
    for i in range(len(test_pos) + len(hires_test_pos)):
        test_labels.append(1)
    for i in range(len(test_neg) + len(hires_test_neg)):
        test_labels.append(0)
    
    train_dataset = ImageDataset(train_data, train_labels)
    val_dataset = ImageDataset(val_data, val_labels)
    # test_dataset = ImageDataset(test_data, test_labels)
    
    
    batch_size = 32
    
    train_data = DataLoader(
        dataset=train_dataset,
        batch_size=batch_size,
        # batch_size = len(train_dataset),
        shuffle=True,
        drop_last=False,
        pin_memory=True if device.type == 'cuda' else False 
    )
    
    val_data = DataLoader(
        dataset=val_dataset,
        batch_size=batch_size,
        # batch_size = len(val_dataset),
        shuffle=False,
        drop_last=False,
        pin_memory=True if device.type == 'cuda' else False
    )


    test_pos = pd.concat([test_pos, hires_test_pos])
    test_neg = pd.concat([test_neg, hires_test_neg])
    
    # print(f'Data splits created. Train size: {len(train_data) * batch_size} Val Size: {len(val_data) * batch_size} Test Size: {len(test_data)}')
    print(f'Data splits created. Train size: {len(train_data) * batch_size} Val Size: {len(val_data) * batch_size} Test Size: {len(test_data)}')
    
    
    
    
    import torch.nn as nn
    import torch.nn.functional as F
    
    
    
    # class HARPSNet(nn.Module):
    #     def __init__(self):
    #         super().__init__()
    #         self.encoder = nn.Sequential(
    #             # Block 1: Input 1x128x128 -> Output 16x64x64
    #             nn.Conv2d(1, 16, kernel_size=3, padding=1),
    #             nn.BatchNorm2d(16),
    #             nn.ReLU(inplace=True),
    #             nn.MaxPool2d(2, 2),
    
    #             # Block 2: Input 16x64x64 -> Output 32x32x32
    #             nn.Conv2d(16, 32, kernel_size=3, padding=1),
    #             nn.BatchNorm2d(32),
    #             nn.ReLU(inplace=True),
    #             nn.MaxPool2d(2, 2),
    
    #             # Block 3: Input 32x32x32 -> Output 64x16x16
    #             nn.Conv2d(32, 64, kernel_size=3, padding=1),
    #             nn.BatchNorm2d(64),
    #             nn.ReLU(inplace=True),
    #             nn.MaxPool2d(2, 2),
                
    #             # Block 4: Input 64x16x16 -> Output 128x8x8
    #             nn.Conv2d(64, 128, kernel_size=3, padding=1),
    #             nn.BatchNorm2d(128),
    #             nn.ReLU(inplace=True),
    #             nn.MaxPool2d(2, 2),
    
    #             nn.Flatten(),
                
    #             # The feature map is now 128 channels * 8 * 8 spatial size = 8192
    #             # This is the same size as before, but it contains much richer, 
    #             # high-level semantic info rather than low-level noise.
    #             nn.Linear(128 * 8 * 8, 512),
    #             nn.ReLU(inplace=True),
    #             # nn.Dropout(0.5), # Increased dropout for the larger layer 
                
    #             # Optional: Add a second linear layer if you have many classes
    #             # nn.Linear(512, 256) 
    #             # Note: If this is for classification, you'd usually add your final 
    #             # class projection here (e.g., nn.Linear(256, num_classes))
    #         )
    
    #     def forward(self, x):
    #         x = self.encoder(x)
    #         x = F.normalize(x, p = 2, dim = 1)
    #         return x

    #         # return self.encoder(x)




    class SelfAttentionBlock(nn.Module):
        def __init__(self, in_channels, spatial_size):
            super().__init__()
            self.norm = nn.LayerNorm(in_channels)
            # Suggestion: Use more than 1 head (e.g., 4 or 8) to capture different relationships
            self.mha = nn.MultiheadAttention(embed_dim=in_channels, num_heads=8, batch_first=True)
            
            # Learnable Positional Encoding: Sequence Length (H*W) x Channels
            self.pos_embedding = nn.Parameter(torch.randn(1, spatial_size * spatial_size, in_channels))
            
            # SAGAN style gating
            self.scale = nn.Parameter(torch.zeros(1))
    
        def forward(self, x):
            bs, c, h, w = x.shape
            
            # Reshape to (Batch, Seq_Len, Dim)
            x_flat = x.reshape(bs, c, h * w).transpose(1, 2)
            
            # Apply Norm
            x_norm = self.norm(x_flat)
            
            # Add Positional Embedding (Broadcasting across batch)
            x_norm = x_norm + self.pos_embedding
            
            # Apply Attention
            # key_padding_mask is not needed here as images are fixed size
            attn_out, _ = self.mha(x_norm, x_norm, x_norm)
            
            # Reshape back to (Batch, Dim, H, W)
            attn_out = attn_out.transpose(1, 2).reshape(bs, c, h, w)
            
            # Residual connection with Scale
            return self.scale * attn_out + x

    class HARPSNet(nn.Module):
        def __init__(self):
            super().__init__()
            self.encoder = nn.Sequential(
                # Block 1: 1 -> 16
                nn.Conv2d(1, 16, kernel_size=3, padding=1),
                nn.BatchNorm2d(16),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2, 2),
        
                # Block 2: 16 -> 32
                nn.Conv2d(16, 32, kernel_size=3, padding=1),
                nn.BatchNorm2d(32),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2, 2),
        
                # Block 3: 32 -> 64
                nn.Conv2d(32, 64, kernel_size=3, padding=1),
                nn.BatchNorm2d(64),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2, 2),
                
                # Block 4: 64 -> 128 (Output size 8x8)
                nn.Conv2d(64, 128, kernel_size=3, padding=1),
                nn.BatchNorm2d(128),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2, 2),
            )
    
            # Initialize Attention Block
            # We know the output is 128 channels and size 8x8
            self.attention = SelfAttentionBlock(in_channels=128, spatial_size=8)
    
            self.flatten = nn.Flatten()
            self.linear = nn.Linear(128 * 8 * 8, 512)
            self.relu = nn.ReLU(inplace=True)
            self.do = nn.Dropout(0.5)
            # Add final classification layer if needed
            # self.final = nn.Linear(512, num_classes)
    
        def forward(self, x):
            x = self.encoder(x)
            
            # Apply Attention
            x = self.attention(x)
            
            x = self.flatten(x)
            x = self.linear(x)
            x = self.relu(x)
            x = self.do(x)
            
            return x
    
    
    
    
    model = HARPSNet()
    
    
    
    
    
    !pip install -q pytorch-metric-learning
    
    
    
    
    import torch.optim
    from pytorch_metric_learning import losses, miners
    from torch.optim.lr_scheduler import ReduceLROnPlateau
    
    
    
    model = model.to(device)
    
    
    
    # optimizer = torch.optim.SGD(model.parameters(), lr=0.0001, momentum=0.9)
    # optimizer = torch.optim.SGD(model.parameters(), lr=0.00001, momentum=0.9, weight_decay = 0.0001)
    # optimizer = torch.optim.Adam(model.parameters(), lr = 0.00001, weight_decay = 0.0001)
    # optimizer = torch.optim.AdamW(model.parameters(), lr = 0.0001, weight_decay = 0.01)
    
    # criterion = nn.BCELoss()
    # criterion = ContrastiveLoss()
    margin = 0.7
    criterion = nn.TripletMarginLoss(margin = margin, p = 2)
    
    # margin = 14.32394
    # margin = 1
    # criterion = losses.ArcFaceLoss(num_classes = 2, embedding_size = 256, margin = margin, scale = 64)
    optimizer = torch.optim.AdamW(model.parameters(), lr = 0.001, weight_decay = 0.1)
    
    
    
    miner = miners.TripletMarginMiner(
        margin=margin,
        type_of_triplets="semihard"
    )
    
    scheduler = ReduceLROnPlateau(
        optimizer,
        mode='min',         
                            
        factor=0.5,         
        patience=10,         
        min_lr=0.001,     
        verbose=True        
    )
    
    
    
    train_losses = []
    val_losses = [100]
    epochs = 500
    patience = 10
    patience_counter = 0
    
    for epoch in range(epochs):
        epoch_loss = 0
        model.train()
        
        for images, labels in train_data:
            images = images.float().to(device)
            labels = labels.to(device)
    
            optimizer.zero_grad()
    
            current_embeddings = model.forward(images.view(-1, 1, 128, 128))
            hard_triplet_indices = miner(current_embeddings, labels)
            if hard_triplet_indices[0].shape[0] == 0:
                # print(f"Epoch {epoch}: No semi-hard triplets found in this batch. Skipping update.")
                continue
    
            emb_anchors = current_embeddings[hard_triplet_indices[0]]
            emb_positives = current_embeddings[hard_triplet_indices[1]]
            emb_negatives = current_embeddings[hard_triplet_indices[2]]
            
            loss = criterion(emb_anchors, emb_positives, emb_negatives)
            # loss = criterion(current_embeddings, labels)
    
            loss.backward()
            optimizer.step()
    
            epoch_loss += loss.item()
    
        avg_train_loss = epoch_loss / len(train_data)
        train_losses.append(avg_train_loss)
    
    
        epoch_loss = 0
        model.eval()
        with torch.no_grad():
            for images, labels in val_data:
                images = images.float().to(device)
                labels = labels.to(device)
    
                current_embeddings = model.forward(images.view(-1, 1, 128, 128))
                hard_triplet_indices = miner(current_embeddings, labels)
                if hard_triplet_indices[0].shape[0] == 0:
                    # print(f"Epoch {epoch}: No semi-hard triplets found in this batch. Skipping.")
                    continue # Skip this batch if no triplets can be formed
        
                emb_anchors = current_embeddings[hard_triplet_indices[0]]
                emb_positives = current_embeddings[hard_triplet_indices[1]]
                emb_negatives = current_embeddings[hard_triplet_indices[2]]
            
                loss = criterion(emb_anchors, emb_positives, emb_negatives)
                # loss = criterion(current_embeddings, labels)
        
                epoch_loss += loss.item()
                
        avg_val_loss = epoch_loss / len(val_data)
        val_losses.append(avg_val_loss)
        
    
        # if epoch % 100 == 0:
        #     print(f'Epoch: {epoch} | Average Train Loss: {avg_train_loss}\t Average Validation Loss: {avg_val_loss}')
    
    
        if avg_val_loss < val_losses[-2]:
            patience_counter = 0
        else:
            patience_counter += 1
    
    
        if avg_val_loss == sorted(val_losses)[0]:
            torch.save(model.state_dict(), 'model.pth')
            # print(f'model saved at: \t{avg_val_loss}')
    
        
        if patience_counter >= patience:
            print("Patience reached")
            break
    
    
    val_losses.pop(0)
    print("Model training finished")

    plt.figure(figsize=(16,9))
    plt.plot(train_losses, c='b', label='Train loss')
    plt.plot(val_losses, c='r', label = 'Validation loss')
    plt.legend()
    plt.grid()
    plt.xlabel('Epochs', fontsize=20)
    plt.ylabel('Loss', fontsize=20)
    plt.title(f'Losses for seed: {seed}')
    
    
    
    
    
    import h5py as h5
    from astropy.timeseries import LombScargle
    
    
    
    def gen_periodogram(star_name):
            try:
                # Filter and clean data
                filtered = harps_df[harps_df['main_id_simbad'] == star_name]
                filtered = filtered.dropna(subset=['drs_bjd', 'drs_ccf_rvc', 'drs_dvrms'])
                
                # Skip if insufficient data
                if len(filtered) < 3:
                    # Filter and clean data
                    filtered = hires_df[hires_df['main_id_simbad'] == star_name]
                    filtered = filtered.dropna(subset=['drs_bjd', 'drs_ccf_rvc', 'drs_dvrms'])
                    
                    # Skip if insufficient data
                    if len(filtered) < 3:
                        return 1, 1  # Placeholder for "not enough data"
                    
                    # Extract arrays with explicit native conversion
                    def to_native_float64(series):
                        arr = np.array(series)
                        if not arr.dtype.isnative:
                            arr = arr.astype(arr.dtype.newbyteorder('='))
                        return arr.astype(np.float64)
                    
                    time = to_native_float64(filtered['drs_bjd'])
                    rad_vel = to_native_float64(filtered['drs_ccf_rvc'])
                    uncertainty = to_native_float64(filtered['drs_dvrms'])
                    
                    # Normalize timestamps for numerical stability
                    time -= np.min(time)
                    
                    # Generate periodogram
                    periodogram = LombScargle(time, rad_vel, uncertainty)
                    frequency, power = periodogram.autopower()
                    
                    # Resample to 1000 points
                    freq_uniform = np.linspace(frequency.min(), frequency.max(), 1000)
                    power_interp = np.interp(freq_uniform, frequency, power)
                    
                    return freq_uniform, power_interp
                    
                
                # Extract arrays with explicit native conversion
                def to_native_float64(series):
                    arr = np.array(series)
                    if not arr.dtype.isnative:
                        arr = arr.astype(arr.dtype.newbyteorder('='))
                    return arr.astype(np.float64)
                
                time = to_native_float64(filtered['drs_bjd'])
                rad_vel = to_native_float64(filtered['drs_ccf_rvc'])
                uncertainty = to_native_float64(filtered['drs_dvrms'])
                
                # Normalize timestamps for numerical stability
                time -= np.min(time)
                
                # Generate periodogram
                periodogram = LombScargle(time, rad_vel, uncertainty)
                frequency, power = periodogram.autopower()
                
                # Resample to 1000 points
                freq_uniform = np.linspace(frequency.min(), frequency.max(), 1000)
                power_interp = np.interp(freq_uniform, frequency, power)
                
                return freq_uniform, power_interp
                
            except Exception as e:
                print(f"Error processing {star_name}: {e}")
                return 1, 1
    
    
    
    stars = list(test_data['star'])
    
    filename = "/kaggle/working/processed_data.h5"
    with h5.File(filename, 'w') as file:
            for star in stars:
                frequency, power = gen_periodogram(star)
                if type(frequency) == int:
                    pass
                else:
                    # star_group = file.create_group(clean_name(star))
                    star_group = file.create_group(star)
                    star_group.create_dataset('frequencies', data = frequency)
                    star_group.create_dataset('power', data = power)
    
    print("Test data h5 file created for Ante's model.")
    #Evaluate on test data using Ante's Model
    !python3 /kaggle/input/datasets/maanav0114/model-and-baselines-evaluation-data/modified_validateRealData.py /kaggle/working/processed_data.h5 /kaggle/input/datasets/maanav0114/model-and-baselines-evaluation-data/top_current_model_trained_on_uneven_data_fully.pth --threshold 0.73 --catalog_path /kaggle/input/datasets/maanav0114/model-and-baselines-evaluation-data/catalog_of_exoplanets.csv
    
    
    preds = pd.DataFrame(pd.read_csv('/kaggle/working/planet_predictions.csv'))
    star_level = pd.DataFrame(columns = preds.columns)
    # test_data = pd.DataFrame(pd.read_csv('/kaggle/input/datasets/maanav0114/model-and-baselines-evaluation-data/test_data.csv')).drop('Unnamed: 0', axis = 1)
    stars = list(set(preds['Star Name']))
    preds['Is True Planet'] = preds['Is True Planet'].astype(int)
    
    for i in range(len(stars)):
        star_level.loc[i, 'Star Name'] = stars[i]
    
    for i in range(len(preds)):
        if preds.iloc[i]['Prediction'] > 0.73:
            preds.loc[i, 'Prediction'] = 1.0
        else:
            preds.loc[i, 'Prediction'] = 0.0
    
        
        # if preds.iloc[i]['Is True Planet'] == True:
        #     preds.loc[i, 'Is True Planet'] = 1.0
        # else:
        #     preds.loc[i, 'Is True Planet'] = 0.0
    
    
    for i in range(len(stars)):
        star = stars[i]
        prediction = min(sum(list(preds[preds['Star Name'] == star]['Prediction'])), 1)
        star_level.loc[i, 'Prediction'] = prediction
    
        # label = sum(list(preds[preds['Star Name'] == star]['Is True Planet']))
        try:
            # label = test_data[test_data['star'] == star].iloc[0][-1]
            for idx in range(len(test_data)):
                if star in test_data.iloc[idx][0]:
                    label = test_data.iloc[idx][-1]
                    break
        except Exception as e:
            print(star)
            print(test_data[test_data['star'] == star])
        # print(label)
        try:
            star_level.loc[i, 'Is True Planet'] = label
        except Exception as e:
            print(e)
            print(test_data.iloc[idx][0])
    
    
    
    true_pos = 0
    true_neg = 0
    false_pos = 0
    false_neg = 0
    
    for i in range(len(star_level)):
        if star_level.iloc[i]['Prediction'] == star_level.iloc[i]['Is True Planet']:
            if star_level.iloc[i]['Is True Planet'] == 0.0:
                true_neg += 1
            elif star_level.iloc[i]['Is True Planet'] == 1.0:
                true_pos += 1
        else:
            if star_level.iloc[i]['Is True Planet'] == 0.0:
                false_pos += 1
            elif star_level.iloc[i]['Is True Planet'] == 1.0:
                false_neg += 1
    
    
    accuracy = (true_pos + true_neg) / (true_pos + true_neg + false_pos + false_neg)
    precision = true_pos / (true_pos + false_pos)
    recall = true_pos / (true_pos + false_neg)
    f1 = (2 * precision * recall) / (precision + recall)
    
    diplomski_accuracies.append(accuracy)
    diplomski_precisions.append(precision)
    diplomski_recalls.append(recall)
    diplomski_f1s.append(f1)
    
    
    
    
    
    preds = pd.DataFrame(pd.read_csv('/kaggle/working/planet_predictions.csv'))
    pred_stars = list(set(preds['Star Name']))
    
    # test_df = pd.read_csv('/kaggle/working/test_data.csv', converters={'image': parse_image}).drop('Unnamed: 0', axis = 1)
    temp = pd.DataFrame(columns = test_data.columns)
    stars = list(test_data['star'])
    for pred_star in pred_stars:
        # print("".join(pred_star.split()))
        for star in stars:
            if "".join(pred_star.split()) == "".join(star.split()):
            # score = fuzz.ratio(pred_star, star)
            # if score >= 91:
                temp = pd.concat([temp, test_data[test_data['star'] == star]])
                break
    
    test_pos = temp[temp['label'] == 1]
    test_neg = temp[temp['label'] == 0]
    
    
    

    del model
    model = HARPSNet().to(device)
    model.load_state_dict(torch.load('/kaggle/working/model.pth'))
    model.eval()
    
    pos_embeddings = []
    
    with torch.no_grad():
        for i in range(len(train_pos)):
            image = torch.tensor(train_pos.iloc[i]['image']).float().to(device).view(1, -1, 128, 128)
            embedding = model.forward(image)
            pos_embeddings.append(embedding.detach())
    
    stacked_embeddings = torch.cat(pos_embeddings, dim=0)
    mean_vector = torch.mean(stacked_embeddings, dim=0)
    pos_prototype = F.normalize(mean_vector, p=2, dim=0).view(1, -1)
    
    
    neg_embeddings = []
    
    with torch.no_grad():
        for i in range(len(train_neg)):
            image = torch.tensor(train_neg.iloc[i]['image']).float().to(device).view(1, -1, 128, 128)
            embedding = model.forward(image)
            neg_embeddings.append(embedding.detach())
    
    stacked_embeddings = torch.cat(neg_embeddings, dim=0)
    mean_vector = torch.mean(stacked_embeddings, dim=0)
    neg_prototype = F.normalize(mean_vector, p=2, dim=0).view(1, -1)





    # from sklearn.neighbors import KNeighborsClassifier



    # train_embeddings = []
    # train_labels_np = []

    # with torch.no_grad():
    #     # assuming train_data is your DataLoader
    #     for images, labels in train_data:
    #         images = images.float().to(device).view(-1, 1, 128, 128) # Adjust size as needed
    #         emb = model.encoder(images)
    #         emb = emb.view(emb.size(0), -1) # Flatten the embeddings
    #         emb = F.normalize(emb, p=2, dim=1)
    #         train_embeddings.append(emb.cpu().numpy())
    #         train_labels_np.extend(labels.cpu().numpy())

    # train_embeddings = np.vstack(train_embeddings)
    # train_labels_np = np.array(train_labels_np)

    # # 2. Fit k-NN Classifier
    # knn = KNeighborsClassifier(n_neighbors=3, metric='cosine') #nga
    # knn.fit(train_embeddings, train_labels_np)






    true_pos = 0
    true_neg = 0
    false_pos = 0
    false_neg = 0



    for idx in range(len(test_pos)):
        test_image = torch.tensor(test_pos.iloc[idx]['image']).float().to(device).view(1, -1, 128, 128)

        with torch.no_grad():
            embedding = model.forward(test_image)

        pos_sim = F.cosine_similarity(embedding, pos_prototype)
        neg_sim = F.cosine_similarity(embedding, neg_prototype)

        if pos_sim > neg_sim:
            true_pos += 1
        else:
            false_neg += 1

    for idx in range(len(test_neg)):
        test_image = torch.tensor(test_neg.iloc[idx]['image']).float().to(device).view(1, -1, 128, 128)

        with torch.no_grad():
            embedding = model.forward(test_image)

        pos_sim = F.cosine_similarity(embedding, pos_prototype)
        neg_sim = F.cosine_similarity(embedding, neg_prototype)

        if pos_sim > neg_sim:
            false_pos += 1
        else:
            true_neg += 1



    # test_embeddings = []
    # test_labels_real = []

    # for idx in range(len(test_pos)):
    #     test_image = torch.tensor(test_pos.iloc[idx]['image']).float().to(device).view(1, -1, 128, 128)

    #     with torch.no_grad():
    #         emb = model.encoder(test_image)
    #         emb = emb.view(emb.size(0), -1) # Flatten the embeddings

    #     test_embeddings.append(emb.cpu().numpy())
    #     test_labels_real.append(1)


    # for idx in range(len(test_neg)):
    #     test_image = torch.tensor(test_neg.iloc[idx]['image']).float().to(device).view(1, -1, 128, 128)

    #     with torch.no_grad():
    #         emb = model.encoder(test_image)
    #         emb = emb.view(emb.size(0), -1) # Flatten the embeddings

    #     test_embeddings.append(emb.cpu().numpy())
    #     test_labels_real.append(0)

    # test_embeddings = np.vstack(test_embeddings)

    # predictions = knn.predict(test_embeddings)

    # for idx in range(len(test_labels_real)):
    #     label = test_labels_real[idx]
    #     pred = predictions[idx]

    #     if label == 1:
    #         if pred == 1:
    #             true_pos += 1
    #         else:
    #             false_neg += 1
    #     else:
    #         if pred == 0:
    #             true_neg += 1
    #         else:
    #             false_pos += 1

    
    
    
    accuracy = (true_pos + true_neg) / (true_pos + true_neg + false_pos + false_neg) if (true_pos + true_neg + false_pos + false_neg) > 0 else 0
    precision = true_pos / (true_pos + false_pos) if (true_pos + false_pos) > 0 else 0
    recall = true_pos / (true_pos + false_neg) if (true_pos + false_neg) > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    siamese_accuracies.append(accuracy)
    siamese_precisions.append(precision)
    siamese_recalls.append(recall)
    siamese_f1s.append(f1)
    
    print("Model Evaluation Complete")
    
    
    
    
    
    def gen_pg(star_name):
        try:
            # Filter and clean data
            filtered = harps_df[harps_df['main_id_simbad'] == star_name]
            filtered = filtered.dropna(subset=['drs_bjd', 'drs_ccf_rvc', 'drs_dvrms'])
            
            # Skip if insufficient data
            if len(filtered) < 3:
                filtered = hires_df[hires_df['main_id_simbad'] == star_name]
                filtered = filtered.dropna(subset=['drs_bjd', 'drs_ccf_rvc', 'drs_dvrms'])
                
                # Skip if insufficient data
                if len(filtered) < 3:
                    return 1, 1  # Placeholder for "not enough data"
                
                # Extract arrays with explicit native conversion
                def to_native_float64(series):
                    arr = np.array(series)
                    if not arr.dtype.isnative:
                        arr = arr.astype(arr.dtype.newbyteorder('='))
                    return arr.astype(np.float64)
                
                time = to_native_float64(filtered['drs_bjd'])
                rad_vel = to_native_float64(filtered['drs_ccf_rvc'])
                uncertainty = to_native_float64(filtered['drs_dvrms'])
                
                # Normalize timestamps for numerical stability
                time -= np.min(time)
                
                # Generate periodogram
                periodogram = LombScargle(time, rad_vel, uncertainty)
                frequency, power = periodogram.autopower()
                
                # Resample to 1000 points
                freq_uniform = np.linspace(frequency.min(), frequency.max(), 990)
                power_interp = np.interp(freq_uniform, frequency, power)
                
                return freq_uniform, power_interp
            
            # Extract arrays with explicit native conversion
            def to_native_float64(series):
                arr = np.array(series)
                if not arr.dtype.isnative:
                    arr = arr.astype(arr.dtype.newbyteorder('='))
                return arr.astype(np.float64)
            
            time = to_native_float64(filtered['drs_bjd'])
            rad_vel = to_native_float64(filtered['drs_ccf_rvc'])
            uncertainty = to_native_float64(filtered['drs_dvrms'])
            
            # Normalize timestamps for numerical stability
            time -= np.min(time)
            
            # Generate periodogram
            periodogram = LombScargle(time, rad_vel, uncertainty)
            frequency, power = periodogram.autopower()
            
            # Resample to 1000 points
            freq_uniform = np.linspace(frequency.min(), frequency.max(), 990)
            power_interp = np.interp(freq_uniform, frequency, power)
            
            return freq_uniform, power_interp
            
        except Exception as e:
            print(f"Error processing {star_name}: {e}")
            return 1, 1
    
    
    def gen_peak_pow(power):
        peak_pow = max(list(power))
        peak = list(power).index(max(list(power)))
    
        return peak, peak_pow
    
    
    
    
    
    
    from tensorflow import keras
    from keras.models import Model
    # from keras.utils.vis_utils import plot_model
    from keras.utils import plot_model #I had to make this change, as the import statement above is not supported by current keras version
    from keras.models import load_model
    import tensorflow as tf
    
    
    
    model = load_model("/kaggle/input/datasets/maanav0114/model-and-baselines-evaluation-data/exoplANNET_trained.h5")
    
    def run_inference(star_name):
        freq, power = gen_pg(star_name)
        peak, peak_pow = gen_peak_pow(power)
    
        pg = np.transpose(np.array([power]))
        peak_pow = np.transpose(np.array([[peak, peak_pow]]))
    
        pg = np.expand_dims(pg, 0)
        peak_pow = np.expand_dims(peak_pow, 0)
    
        output = model.predict([pg, peak_pow])
        # print(output)
        if output < 0.77:
            output = 0
        else:
            output = 1
    
        return output
    
    
    
    predictions = {}
    for star in pred_stars:
        output = run_inference(star)
        predictions[star] = output
    
    
    
    
    
    true_pos = 0
    true_neg = 0
    false_pos = 0
    false_neg = 0
    
    
    for star in predictions:
        prediction = predictions[star]
        label = int(test_data[test_data['star'] == star]['label'])
    
        if label == 1:
            if prediction == 1:
                true_pos += 1
            else:
                false_neg += 1
        else:
            if prediction == 1:
                false_pos += 1
            else:
                true_neg += 1
    
    
    accuracy = (true_pos + true_neg) / (true_pos + true_neg + false_pos + false_neg)
    precision = true_pos / (true_pos + false_pos)
    recall = true_pos / (true_pos + false_neg)
    f1 = (2 * precision * recall) / (precision + recall)
    
    ExoplANNET_accuracies.append(accuracy)
    ExoplANNET_precisions.append(precision)
    ExoplANNET_recalls.append(recall)
    ExoplANNET_f1s.append(f1)
    
    print('ExoplANNET evaluation complete.')
    # break

In [ ]:
def std_dev(metric):
    mean = sum(metric)/len(metric)

    sq_diffs = 0
    for data in metric:
        diff = (data - mean) ** 2
        sq_diffs += diff
    sq_diffs = sq_diffs / len(metric)

    return sq_diffs ** 0.5

In [ ]:
siamese_mean_accuracy = sum(siamese_accuracies)/len(siamese_accuracies)
siamese_mean_precision = sum(siamese_precisions)/len(siamese_precisions)
siamese_mean_recall = sum(siamese_recalls)/len(siamese_recalls)
siamese_mean_f1 = sum(siamese_f1s)/len(siamese_f1s)

print(f'siamese_mean_accuracy: {siamese_mean_accuracy}\tsiamese_accuracy_std_dev: {std_dev(siamese_accuracies)}')
print(f'siamese_mean_precision: {siamese_mean_precision}\tsiamese_precision_std_dev: {std_dev(siamese_precisions)}')
print(f'siamese_mean_recall: {siamese_mean_recall}\tsiamese_recall_std_dev: {std_dev(siamese_recalls)}')
print(f'siamese_mean_f1: {siamese_mean_f1}\tsiamese_f1_std_dev: {std_dev(siamese_f1s)}')

In [ ]:
diplomski_mean_accuracy = sum(diplomski_accuracies)/len(diplomski_accuracies)
diplomski_mean_precision = sum(diplomski_precisions)/len(diplomski_precisions)
diplomski_mean_recall = sum(diplomski_recalls)/len(diplomski_recalls)
diplomski_mean_f1 = sum(diplomski_f1s)/len(diplomski_f1s)

print(f'diplomski_mean_accuracy: {diplomski_mean_accuracy}\tdiplomski_accuracy_std_dev: {std_dev(diplomski_accuracies)}')
print(f'diplomski_mean_precision: {diplomski_mean_precision}\tdiplomski_precision_std_dev: {std_dev(diplomski_precisions)}')
print(f'diplomski_mean_recall: {diplomski_mean_recall}\tdiplomski_recall_std_dev: {std_dev(diplomski_recalls)}')
print(f'diplomski_mean_f1: {diplomski_mean_f1}\tdiplomski_f1_std_dev: {std_dev(diplomski_f1s)}')

In [ ]:
ExoplANNET_mean_accuracy = sum(ExoplANNET_accuracies)/len(ExoplANNET_accuracies)
ExoplANNET_mean_precision = sum(ExoplANNET_precisions)/len(ExoplANNET_precisions)
ExoplANNET_mean_recall = sum(ExoplANNET_recalls)/len(ExoplANNET_recalls)
ExoplANNET_mean_f1 = sum(ExoplANNET_f1s)/len(ExoplANNET_f1s)

print(f'ExoplANNET_mean_accuracy: {ExoplANNET_mean_accuracy}\tExoplANNET_accuracy_std_dev: {std_dev(ExoplANNET_accuracies)}')
print(f'ExoplANNET_mean_precision: {ExoplANNET_mean_precision}\tExoplANNET_precision_std_dev: {std_dev(ExoplANNET_precisions)}')
print(f'ExoplANNET_mean_recall: {ExoplANNET_mean_recall}\tExoplANNET_recall_std_dev: {std_dev(ExoplANNET_recalls)}')
print(f'ExoplANNET_mean_f1: {ExoplANNET_mean_f1}\tExoplANNET_f1_std_dev: {std_dev(ExoplANNET_f1s)}')